# Thí nghiệm - Helpdesk event log

Logic và kết quả của các mục trong báo cáo (`docs/report.md`). Mỗi phần: câu hỏi → code → kết quả.
Dùng chung các module trong `src/helpdesk_app/`. Cột giữ nguyên tên gốc: `case:concept:name`, `Activity`, `Resource`, `time:timestamp`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pm4py
from helpdesk_app.event_log import load_event_log, traces, CASE, ACTIVITY, RESOURCE, TIMESTAMP

log = load_event_log()
DAY = pd.Timedelta(days=1)
year = log.groupby(CASE)[TIMESTAMP].min().dt.year
print(f"{log[CASE].nunique()} ticket, {len(log)} sự kiện, {log[ACTIVITY].nunique()} loại bước, {log[RESOURCE].nunique()} người")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




4580 ticket, 21348 sự kiện, 14 loại bước, 22 người


## 5.1. Ticket có kẹt hẳn ở Wait không?

Ticket kết thúc tại Wait (không có bước nào sau đó) là ticket kẹt hẳn. Nếu ít, Wait là chờ lâu rồi đi tiếp (điểm nghẽn), không phải deadlock.

In [2]:
last = log.groupby(CASE)[ACTIVITY].last()
end_wait = last == "Wait"
print(f"kết thúc tại Wait: {end_wait.sum()} / {len(last)} ticket ({end_wait.mean():.1%})")
print("bước cuối của các ticket:", last.value_counts().head(4).to_dict())
pd.DataFrame({"ticket": year.value_counts().sort_index(), "kết thúc tại Wait": end_wait.groupby(year).sum()})

kết thúc tại Wait: 8 / 4580 ticket (0.2%)
bước cuối của các ticket: {'Closed': 4557, 'Resolve ticket': 10, 'Wait': 8, 'Require upgrade': 3}


,ticket,kết thúc tại Wait
time:timestamp,,
2010,977,0
2011,1518,0
2012,1412,1
2013,673,7


In [3]:
from helpdesk_app.waiting import wait_durations, days_to_resolve
waits = wait_durations(log)
resolve = days_to_resolve(log)
print(f"mỗi lần Wait: trung vị {waits.median():.1f} ngày, trung bình {waits.mean():.1f} ngày")
resolve.groupby("has_wait")["days"].agg(ticket="size", trung_vị="median", trung_bình="mean", lâu_nhất="max").round(1)

mỗi lần Wait: trung vị 6.9 ngày, trung bình 9.6 ngày


,ticket,trung_vị,trung_bình,lâu_nhất
has_wait,,,,
False,3378,2.8,8.0,58.7
True,1189,19.1,19.9,44.9


Dùng trung vị: ticket không có Wait bị lệch (đa số xong rất nhanh, vài ticket kéo dài), nên trung bình bị kéo lên.

In [4]:
no_wait = resolve.loc[~resolve["has_wait"], "days"]
print(f"ticket không có Wait xong trong 3 ngày: {(no_wait <= 3).mean():.0%}")

ticket không có Wait xong trong 3 ngày: 52%


## 5.2–5.3. Năm 2013 quy trình có đổi không? (Graph2Vec)

Mỗi năm dựng một BPMN (Inductive Miner, 5 đường đi phổ biến nhất). Graph2Vec: đổi tên từng node theo hàng xóm (Weisfeiler–Lehman) thành các "từ",
coi mỗi BPMN là một văn bản rồi học vector bằng Doc2Vec. Hai năm có vector gần nhau (cosine cao) thì quy trình giống nhau.

In [5]:
from helpdesk_app.waiting import wait_share_by_year
wait_share_by_year(log).round(3)

,tickets,wait_share
year,,
2010,977,0.212
2011,1518,0.243
2012,1412,0.209
2013,673,0.542


In [6]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from helpdesk_app.process_model import discover_bpmn, bpmn_graph

tr = traces(log)
graphs = {}
for y in [2010, 2011, 2012, 2013]:
    cases = year[year == y].index
    bpmn, share = discover_bpmn(log[log[CASE].isin(cases)])
    graphs[y] = g = bpmn_graph(bpmn)
    print(y, f"5 đường đi phủ {share:.0%} |", "bước:", sorted(n for n, k in g.nodes(data="kind") if k == "Task"))
    for v, s in tr[tr.index.isin(cases)].value_counts(normalize=True).head(2).items():
        print("    ", f"{s:.0%}", " → ".join(v))

2010 5 đường đi phủ 81% | bước: ['Assign seriousness', 'Closed', 'Insert ticket', 'Resolve ticket', 'Take in charge ticket', 'Wait']
     55% Assign seriousness → Take in charge ticket → Resolve ticket → Closed
     10% Assign seriousness → Take in charge ticket → Wait → Take in charge ticket → Resolve ticket → Closed
2011 5 đường đi phủ 78% | bước: ['Assign seriousness', 'Closed', 'Resolve ticket', 'Take in charge ticket', 'Wait']
     52% Assign seriousness → Take in charge ticket → Resolve ticket → Closed
     7% Assign seriousness → Take in charge ticket → Wait → Take in charge ticket → Resolve ticket → Closed
2012 5 đường đi phủ 88% | bước: ['Assign seriousness', 'Closed', 'Resolve ticket', 'Take in charge ticket', 'Wait']
     61% Assign seriousness → Take in charge ticket → Resolve ticket → Closed
     15% Assign seriousness → Take in charge ticket → Wait → Resolve ticket → Closed
2013 5 đường đi phủ 70% | bước: ['Assign seriousness', 'Closed', 'Require upgrade', 'Resolve ticket

In [7]:
from helpdesk_app.bpmn_embedding import graph2vec_similarity

similarity, max_std = graph2vec_similarity(graphs)
print("độ lệch chuẩn giữa 5 seed (lớn nhất):", round(max_std, 3))
similarity.round(2)

độ lệch chuẩn giữa 5 seed (lớn nhất): 0.043


,2010,2011,2012,2013
2010,1.00,0.25,0.25,0.21
2011,0.25,1.00,0.50,0.47
2012,0.25,0.50,1.00,0.52
2013,0.21,0.47,0.52,1.00


## 5.4. Thời gian theo quý: từ quý 4/2012 mọi ticket chậm hơn

Theo quý mở ticket: trung vị số ngày đến Resolve ticket (có / không có Wait), chờ người nhận
(Assign seriousness → Take in charge ticket) và chờ đóng (Resolve ticket → Closed).

In [8]:
from helpdesk_app.waiting import by_quarter
by_quarter(log).round(2)

,tickets,wait_share,days_with_wait,days_without_wait,days_to_take,days_to_close
quarter,,,,,,
2010Q1,154,0.12,22.02,7.31,0.06,29.96
2010Q2,231,0.20,17.29,4.96,1.02,29.84
2010Q3,275,0.24,17.67,1.99,0.18,33.00
2010Q4,317,0.19,14.90,1.76,0.28,31.55
2011Q1,369,0.25,16.82,1.86,0.00,31.86
2011Q2,371,0.31,13.16,1.02,0.00,31.06
2011Q3,300,0.15,14.80,2.03,0.87,31.00
2011Q4,478,0.22,16.27,1.20,0.71,31.90
2012Q1,491,0.16,17.94,1.02,0.03,32.11


Tách log tại quý 4/2012: trung vị số ngày đến Resolve (có / không Wait, và có Wait khi nhắc sau 7 ngày), mỗi lần Wait, và các khoảng chờ chính.

In [9]:
import pandas as pd
from helpdesk_app.event_log import ACTIVITY, CASE, TIMESTAMP
from helpdesk_app.waiting import days_to_resolve, _with_next_gap

opened = log.groupby(CASE)[TIMESTAMP].min()
new = opened[opened >= pd.Timestamp("2012-10-01", tz="UTC")].index
asis, tobe = days_to_resolve(log), days_to_resolve(log, 7)
events = _with_next_gap(log).assign(next=lambda e: e.groupby(CASE)[ACTIVITY].shift(-1))
has_wait = asis["has_wait"]
rows = {}
for name, era in [("trước quý 4/2012", ~asis.index.isin(new)), ("từ quý 4/2012", asis.index.isin(new))]:
    ev = events[events[CASE].isin(asis.index[era])]
    gap = lambda a, b, cases: ev[(ev[ACTIVITY] == a) & (ev["next"] == b) & ev[CASE].isin(cases)]["gap_days"].median()
    w, nw = asis.index[era & has_wait], asis.index[era & ~has_wait]
    rows[name] = {"có Wait": asis.loc[w, "days"].median(), "không Wait": asis.loc[nw, "days"].median(),
                  "có Wait, nhắc sau 7 ngày": tobe.loc[w, "days"].median(),
                  "mỗi lần Wait": ev.loc[ev[ACTIVITY] == "Wait", "gap_days"].median(),
                  "chờ người nhận: không Wait": gap("Assign seriousness", "Take in charge ticket", nw),
                  "chờ người nhận: có Wait": gap("Assign seriousness", "Take in charge ticket", w),
                  "nhận → Resolve trực tiếp: không Wait": gap("Take in charge ticket", "Resolve ticket", nw)}
pd.DataFrame(rows).round(1)

,trước quý 4/2012,từ quý 4/2012
có Wait,16.0,22.8
không Wait,1.8,21.0
"có Wait, nhắc sau 7 ngày",10.0,16.8
mỗi lần Wait,5.1,9.9
chờ người nhận: không Wait,0.1,10.8
chờ người nhận: có Wait,0.2,5.9
nhận → Resolve trực tiếp: không Wait,0.2,6.8


## 6. TO-BE: nhắc khi ticket chờ quá X ngày

### Kiểm tra BPMN TO-BE trước khi triển khai (Petri Net, soundness)

Chuyển BPMN sang Petri Net rồi kiểm tra soundness (không deadlock, luôn về được trạng thái kết thúc).
Thêm một bản vẽ sai (tách bằng XOR nhưng gộp bằng AND) để chắc bộ kiểm tra bắt được lỗi.

In [10]:
from pm4py.objects.bpmn.obj import BPMN
from helpdesk_app.process_model import to_be_graph

def build_bpmn(g, and_join=None):
    bpmn, nodes = BPMN(), {}
    for n, d in g.nodes(data=True):
        k = d["kind"]
        if k == "Start": o = BPMN.StartEvent(name=n)
        elif k == "End": o = BPMN.EndEvent(name=n)
        elif k == "Task": o = BPMN.Task(name=n)
        elif n == and_join: o = BPMN.ParallelGateway(name=n, gateway_direction=BPMN.Gateway.Direction.CONVERGING)
        else: o = BPMN.ExclusiveGateway(name=n, gateway_direction=BPMN.Gateway.Direction.DIVERGING if g.out_degree(n) > 1
                                        else BPMN.Gateway.Direction.CONVERGING)
        nodes[n] = o; bpmn.add_node(o)
    for a, b in g.edges:
        f = BPMN.SequenceFlow(nodes[a], nodes[b]); bpmn.add_flow(f); nodes[a].add_out_arc(f); nodes[b].add_in_arc(f)
    return bpmn

for name, bpmn in [("TO-BE", build_bpmn(to_be_graph())), ("TO-BE vẽ sai (gộp bằng AND)", build_bpmn(to_be_graph(), and_join="XOR 2"))]:
    net, im, fm = pm4py.convert_to_petri_net(bpmn)
    print(f"{name}: {len(net.places)} place, {len(net.transitions)} transition | sound: {pm4py.check_soundness(net, im, fm)[0]}")

TO-BE: 7 place, 8 transition | sound: True
TO-BE vẽ sai (gộp bằng AND): 7 place, 6 transition | sound: False


### Mô phỏng: nhắc sau X ngày thì nhanh hơn bao nhiêu?

Thời gian chờ = `time:timestamp` của bước sau Wait − `time:timestamp` của Wait. Lần chờ dài hơn X ngày bị cắt còn X
(giả định: nhắc xong ticket đi tiếp ngay); các khoảng khác giữ nguyên. Đo số ngày từ bước đầu đến `Resolve ticket`.
Code: `waiting.days_to_resolve(log, remind_after=X)`.

In [11]:
rows = []
for x in [None, 3, 7, 10, 14]:
    t = days_to_resolve(log, remind_after=x)
    rows.append({"nhắc khi chờ quá": "không nhắc" if x is None else f"{x} ngày",
                 "lần Wait bị nhắc": None if x is None else f"{(waits > x).mean():.0%}",
                 "ticket có Wait: trung vị ngày đến Resolve": round(t.loc[t["has_wait"], "days"].median(), 1)})
pd.DataFrame(rows)

,nhắc khi chờ quá,lần Wait bị nhắc,ticket có Wait: trung vị ngày đến Resolve
0,không nhắc,NaN,19.1
1,3 ngày,66%,9.8
2,7 ngày,48%,13.3
3,10 ngày,38%,15.2
4,14 ngày,28%,16.8


## 7. Dự đoán ticket đang chạy

Chia dữ liệu: xếp ticket theo ngày mở; 80% đầu để học, 20% cuối để kiểm tra; bỏ ticket vắt qua mốc.

In [12]:
from helpdesk_app.prediction import split_by_time
train_ids, test_ids, cut = split_by_time(log)
span = log.groupby(CASE)[TIMESTAMP].min()
print(f"mốc chia {cut.date()} | học {len(train_ids)} ticket | kiểm tra {len(test_ids)} ticket | bỏ {len(span) - len(train_ids) - len(test_ids)}")
print("năm của ticket kiểm tra:", span.loc[test_ids].dt.year.value_counts().sort_index().to_dict())

mốc chia 2012-07-30 | học 3415 ticket | kiểm tra 916 ticket | bỏ 249
năm của ticket kiểm tra: {2012: 243, 2013: 673}


### 7.2. Ticket nào hay rẽ vào Wait? (Decision Tree)

Tại điểm rẽ sau `Take in charge ticket`: mỗi ticket một dòng thông tin có lúc nhận (người nhận, thứ, giờ, số bước trước,
thời gian chờ đến lúc nhận). Cây độ sâu 3 để đọc được luật "nếu… thì…".

In [13]:
from sklearn.tree import export_text
from helpdesk_app.prediction import WaitTree, build_samples

samples = build_samples(log)
tr_s, te_s = samples[samples[CASE].isin(train_ids)], samples[samples[CASE].isin(test_ids)]
wait_tree = WaitTree(tr_s)
print(f"tỷ lệ Wait: tập học {tr_s.waits_later.mean():.1%} | tập kiểm tra {te_s.waits_later.mean():.1%}")
print(export_text(wait_tree.tree, feature_names=wait_tree.names, show_weights=True))

tỷ lệ Wait: tập học 22.4% | tập kiểm tra 50.2%
|--- resource_Value 9 <= 0.50
|   |--- resource_Value 8 <= 0.50
|   |   |--- log_hours_before <= 1.00
|   |   |   |--- weights: [881.00, 151.00] class: False
|   |   |--- log_hours_before >  1.00
|   |   |   |--- weights: [1175.00, 328.00] class: False
|   |--- resource_Value 8 >  0.50
|   |   |--- log_hours_before <= 0.01
|   |   |   |--- weights: [141.00, 75.00] class: False
|   |   |--- log_hours_before >  0.01
|   |   |   |--- weights: [100.00, 38.00] class: False
|--- resource_Value 9 >  0.50
|   |--- log_hours_before <= 3.94
|   |   |--- log_hours_before <= 0.71
|   |   |   |--- weights: [60.00, 40.00] class: False
|   |   |--- log_hours_before >  0.71
|   |   |   |--- weights: [86.00, 37.00] class: False
|   |--- log_hours_before >  3.94
|   |   |--- weights: [60.00, 53.00] class: False



In [14]:
leaf_tr, leaf_te = wait_tree.tree.apply(wait_tree.features(tr_s)), wait_tree.tree.apply(wait_tree.features(te_s))
pd.DataFrame({"học: ticket": pd.Series(leaf_tr).value_counts(), "học: Wait": tr_s.waits_later.groupby(leaf_tr).mean().round(2),
              "kiểm tra: ticket": pd.Series(leaf_te).value_counts(), "kiểm tra: Wait": te_s.waits_later.groupby(leaf_te).mean().round(2)}).rename_axis("lá")

,học: ticket,học: Wait,kiểm tra: ticket,kiểm tra: Wait
lá,,,,
3,1032,0.15,182.0,0.48
4,1503,0.22,543.0,0.47
6,216,0.35,NaN,NaN
7,138,0.28,NaN,NaN
10,100,0.40,11.0,0.55
11,123,0.30,13.0,0.69
12,113,0.47,89.0,0.72


`log_hours_before` = log(1 + số giờ từ lúc mở đến lúc nhận): 1,00 ≈ 1,7 giờ; 3,94 ≈ 2 ngày.

### 7.3. Đoán bước tiếp theo và thời gian còn lại đến Resolve ticket

Mỗi ticket cắt thành nhiều điểm dự đoán: sau bước 1, sau bước 2, … Tại mỗi điểm đoán bước kế tiếp (hoặc kết thúc)
và số ngày còn lại đến lần Resolve ticket đầu tiên (giống mục 5). Điểm đã qua Resolve ticket chỉ chấm bước tiếp theo.
Không đo đến Closed: sau Resolve ticket có khoảng chờ đóng gần như cố định (30 ngày, từ quý 4/2012 là 15 ngày), đoán rất dễ
nên làm mô hình trông tốt hơn thực chất. Mọi mô hình dùng vector bước từ Node2Vec (mục 4), chạy 3 seed.

Code các mô hình ở `src/helpdesk_app/prediction.py` (app dùng cùng code):
- **Baseline:** bước hay đi sau bước hiện tại nhất; trung vị thời gian còn lại theo bước hiện tại.
- **Vector bước (Node2Vec):** BPMN của toàn bộ log → Node2Vec → mỗi loại bước một vector 8 chiều.
- **Gradient Boosting:** ticket gộp thành một dòng số (đếm từng loại bước, vector Node2Vec của bước cuối, số bước, thời gian); không biết thứ tự bước.
- **LSTM:** đọc lần lượt từng bước. **Transformer:** nhìn mọi bước cùng lúc (attention, 2 lớp, 4 head, 64 chiều).
  Cả hai dùng vector Node2Vec cố định, cùng 2 đầu ra; Adam 0,001, batch 128, 30 epoch.

In [15]:
from helpdesk_app.prediction import Predictors, prefixes, remind_by_prediction

results, gateway, reminders = [], [], []
for seed in range(3):
    predictors = Predictors(log, train_ids, seed=seed)
    test_points = prefixes(log, test_ids, predictors.idx)
    score = predictors.score(test_points)
    results += [(m, seed, r["bước tiếp theo: đúng"], r["thời gian còn lại: sai TB (ngày)"]) for m, r in score.iterrows()]
    n, wait_share, guesses = predictors.at_gateway(test_points)
    gateway.append(guesses.rename(seed))
    reminders.append(remind_by_prediction(predictors, log, test_ids).assign(seed=seed))
times = np.array([p[4] for p in test_points])
print(f"điểm dự đoán: {len(test_points)} | có thời gian (chưa Resolve): {(~np.isnan(times)).sum()}, "
      f"thực tế còn lại trung vị {np.nanmedian(times):.1f} ngày")

điểm dự đoán: 4411 | có thời gian (chưa Resolve): 2438, thực tế còn lại trung vị 17.9 ngày


In [16]:
df = pd.DataFrame(results, columns=["mô hình", "seed", "đúng", "sai TB (ngày)"])
summary = df.query("not (`mô hình` == 'Baseline' and seed > 0)").groupby("mô hình", sort=False).agg(đúng=("đúng", "mean"), đúng_std=("đúng", "std"),
                                                 sai_TB=("sai TB (ngày)", "mean"), sai_TB_std=("sai TB (ngày)", "std"))
summary.round(4)

,đúng,đúng_std,sai_TB,sai_TB_std
mô hình,,,,
Baseline,0.7026,NaN,15.3741,NaN
Gradient Boosting,0.7476,0.0020,14.9614,0.0385
LSTM,0.7383,0.0247,15.2496,0.0899
Transformer,0.7415,0.0143,15.1132,0.3769


In [17]:
df.round(4)

,mô hình,seed,đúng,sai TB (ngày)
0,Baseline,0,0.7026,15.3741
1,Gradient Boosting,0,0.7497,14.9219
2,LSTM,0,0.7175,15.3477
3,Transformer,0,0.7452,14.7735
4,Baseline,1,0.7026,15.3741
5,Gradient Boosting,1,0.7456,14.9633
6,LSTM,1,0.7656,15.2301
7,Transformer,1,0.7536,15.5186
8,Baseline,2,0.7026,15.3741
9,Gradient Boosting,2,0.7474,14.9988


Vì sao thời gian còn lại khó đoán: tập học và tập kiểm tra khác giai đoạn (mục 5.4).

In [18]:
from helpdesk_app.waiting import days_to_resolve
resolve = days_to_resolve(log)
pd.DataFrame({n: {"ticket": len(x), "trung vị ngày đến Resolve": x.days.median(), "không Wait": x[~x.has_wait].days.median(),
                  "có Wait": x[x.has_wait].days.median(), "tỷ lệ Wait": x.has_wait.mean()}
              for n, ids in [("học", train_ids), ("kiểm tra", test_ids)] for x in [resolve[resolve.index.isin(ids)]]}).round(2)

,học,kiểm tra
ticket,3413.00,905.00
trung vị ngày đến Resolve,3.69,21.04
không Wait,1.78,18.91
có Wait,15.94,22.79
tỷ lệ Wait,0.21,0.48


### 7.4. Tại điểm rẽ: có đoán được Wait không?

Điểm kiểm tra ngay sau lần Take in charge ticket đầu tiên; đếm số lần mỗi mô hình đoán bước tiếp theo là Wait.

In [19]:
print(f"{n} điểm tại điểm rẽ, thực tế {wait_share:.1%} rẽ vào Wait ({round(n * wait_share)} lần)")
pd.concat(gateway, axis=1).rename_axis("số lần đoán Wait / seed")

838 điểm tại điểm rẽ, thực tế 47.9% rẽ vào Wait (401 lần)


,0,1,2
số lần đoán Wait / seed,,,
Baseline,0,0,0
Gradient Boosting,4,6,3
LSTM,0,0,0
Transformer,1,0,1


### 7.5. Hẹn nhắc theo số ngày dự đoán có tốt hơn hẹn cố định X ngày?

Khi ticket vừa vào Wait, hẹn nhắc sau đúng số ngày mô hình đoán còn lại. So với hẹn cố định X ngày có cùng số lần nhắc.
"Bớt" = số ngày chờ bị cắt khi nhắc, chia đều cho mọi lần Wait (giả định nhắc xong đi tiếp ngay, như mục 6).

In [20]:
rem = pd.concat(reminders)
print(f"lần vào Wait ở tập kiểm tra: {rem['waits'].iloc[0]}")
rem.drop(columns=["waits", "seed"]).groupby("mô hình", sort=False).mean().round(2)

lần vào Wait ở tập kiểm tra: 495


,hẹn (trung vị ngày),lần Wait bị nhắc,bớt / lần Wait (ngày),cố định cùng số lần nhắc: X (ngày),cố định: bớt / lần Wait (ngày)
mô hình,,,,,
Baseline,6.96,0.62,7.02,6.95,7.03
Gradient Boosting,6.10,0.63,7.26,6.55,7.28
LSTM,6.49,0.64,7.09,6.32,7.42
Transformer,6.14,0.63,7.07,6.42,7.38
